In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
WhatNet — Unified Multi-Script Handwriting Recognition
=======================================================
Supports:
  Indic-Brahmic : Devanagari (DHCD), Bengali (cMATERdb3.1.2), Kannada (Kannada-MNIST)
  Germanic      : English (EMNIST-Balanced / EMNIST-ByClass / PG-HWLD)
  Semitic       : Persian (HODA), Arabic (ACHD)
  Logographic   : Japanese (Kuzushiji-49)
  Syllabic      : Thai (Burapha-TH)

Usage:
  python whatnet_multiscript.py --script devanagari
  python whatnet_multiscript.py --script bengali
  python whatnet_multiscript.py --script emnist_balanced
  python whatnet_multiscript.py --script persian
  ... etc.

Only the stem kernel and num_classes change per script.
The encoder, AFC capsule, and gated-fusion head are fully shared / transferable.
"""

import os, time, random, json, argparse
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. SCRIPT REGISTRY
#     Each entry fully describes one dataset.  Add new scripts here only.
# ─────────────────────────────────────────────────────────────────────────────
SCRIPT_CONFIGS = {

    # ── Indic-Brahmic ─────────────────────────────────────────────────────────
    "devanagari": {
        "num_classes":  46,          # 36 consonants + 10 digits
        "image_size":   32,
        "color_mode":   "grayscale",
        "stem_type":    "shirorekha",  # captures the Devanagari top-bar (शिरोरेखा)
        "data_dir":     "./data/DHCD",
        "train_subdir": "Train",
        "test_subdir":  "Test",
        "label_mode":   "int",        # directory-based integer labels
        "invert":       True,         # DHCD: dark strokes on light bg → invert
        "augment_rotation": 10,
        "description":  "Devanagari DHCD — 46 classes",
    },

    "bengali": {
        # cMATERdb3.1.2  →  50 classes (11 digits + 39 consonants)
        # Download: https://code.google.com/archive/p/cmaterdb/
        # Expected structure: data/bengali/Train/<class_folder>/ data/bengali/Test/
        "num_classes":  50,
        "image_size":   32,
        "color_mode":   "grayscale",
        "stem_type":    "shirorekha",  # Bengali also has a top-bar (মাত্রা)
        "data_dir":     "./data/bengali",
        "train_subdir": "Train",
        "test_subdir":  "Test",
        "label_mode":   "int",
        "invert":       True,
        "augment_rotation": 10,
        "description":  "Bengali cMATERdb3.1.2 — 50 classes",
    },

    "kannada": {
        # Kannada-MNIST  →  10 digit classes (0-9)
        # Download: https://github.com/vinayprabhu/Kannada_MNIST
        # Comes as .npz files; set data_dir to the folder containing them.
        "num_classes":  10,
        "image_size":   28,
        "color_mode":   "grayscale",
        "stem_type":    "standard",    # Kannada has rounder, non-top-bar strokes
        "data_dir":     "./data/kannada_mnist",
        "train_subdir": None,          # loaded from .npz (see load_dataset())
        "test_subdir":  None,
        "label_mode":   "npz",
        "npz_train":    "X_train.npy", # adjust to actual file names
        "npz_test":     "X_test.npy",
        "npz_train_lbl":"y_train.npy",
        "npz_test_lbl": "y_test.npy",
        "invert":       False,
        "augment_rotation": 8,
        "description":  "Kannada-MNIST — 10 digit classes",
    },

    # ── Germanic ──────────────────────────────────────────────────────────────
    "emnist_balanced": {
        # EMNIST-Balanced  →  47 classes (digits + upper + lower merged)
        # TF Datasets: tfds.load("emnist/balanced")
        "num_classes":  47,
        "image_size":   28,
        "color_mode":   "grayscale",
        "stem_type":    "horizontal",  # Latin letters dominated by horizontal strokes
        "data_dir":     None,           # loaded via tfds
        "train_subdir": None,
        "test_subdir":  None,
        "label_mode":   "tfds",
        "tfds_name":    "emnist/balanced",
        "invert":       True,          # EMNIST pixels are transposed by convention
        "transpose":    True,          # rotate 90°+flip to correct EMNIST orientation
        "augment_rotation": 8,
        "description":  "EMNIST-Balanced — 47 classes",
    },

    "emnist_byclass": {
        # EMNIST-ByClass  →  62 classes (10 digits + 26 upper + 26 lower)
        "num_classes":  62,
        "image_size":   28,
        "color_mode":   "grayscale",
        "stem_type":    "horizontal",
        "data_dir":     None,
        "train_subdir": None,
        "test_subdir":  None,
        "label_mode":   "tfds",
        "tfds_name":    "emnist/byclass",
        "invert":       True,
        "transpose":    True,
        "augment_rotation": 8,
        "description":  "EMNIST-ByClass — 62 classes",
    },

    "emnist_letters": {
        # EMNIST-Letters  →  26 classes (case-merged letters)
        "num_classes":  26,
        "image_size":   28,
        "color_mode":   "grayscale",
        "stem_type":    "horizontal",
        "data_dir":     None,
        "label_mode":   "tfds",
        "tfds_name":    "emnist/letters",
        "invert":       True,
        "transpose":    True,
        "augment_rotation": 8,
        "description":  "EMNIST-Letters — 26 classes",
    },

    "emnist_digits": {
        # EMNIST-Digits  →  10 digit classes (larger than MNIST)
        "num_classes":  10,
        "image_size":   28,
        "color_mode":   "grayscale",
        "stem_type":    "standard",
        "data_dir":     None,
        "label_mode":   "tfds",
        "tfds_name":    "emnist/digits",
        "invert":       True,
        "transpose":    True,
        "augment_rotation": 8,
        "description":  "EMNIST-Digits — 10 classes",
    },

    "emnist_mnist": {
        # EMNIST-MNIST  →  10 digit classes (MNIST-compatible split)
        "num_classes":  10,
        "image_size":   28,
        "color_mode":   "grayscale",
        "stem_type":    "standard",
        "data_dir":     None,
        "label_mode":   "tfds",
        "tfds_name":    "emnist/mnist",
        "invert":       True,
        "transpose":    True,
        "augment_rotation": 8,
        "description":  "EMNIST-MNIST — 10 classes",
    },

    "pghwld": {
        # PG-HWLD (Pen-Graffiti Handwritten Latin Dataset)
        # Expected: data/pghwld/Train/<class>/ data/pghwld/Test/<class>/
        "num_classes":  36,           # adjust to actual class count in your version
        "image_size":   32,
        "color_mode":   "grayscale",
        "stem_type":    "horizontal",
        "data_dir":     "./data/pghwld",
        "train_subdir": "Train",
        "test_subdir":  "Test",
        "label_mode":   "int",
        "invert":       False,
        "augment_rotation": 10,
        "description":  "PG-HWLD Latin handwriting",
    },

    # ── Semitic ───────────────────────────────────────────────────────────────
    "persian": {
        # HODA Persian Handwritten Digit Dataset  →  10 digit classes
        # Download: http://farsiocr.ir/مجموعه-داده/مجموعه-ارقام-دست‌نویس-هدی/
        # Expected: data/hoda/Train/<class>/ data/hoda/Test/<class>/
        # NOTE: Persian/Arabic are RTL.  The stem_type "rtl" uses both 1×5 and 5×1
        # kernels symmetrically since character joining changes stroke direction.
        "num_classes":  10,
        "image_size":   32,
        "color_mode":   "grayscale",
        "stem_type":    "rtl",
        "data_dir":     "./data/hoda",
        "train_subdir": "Train",
        "test_subdir":  "Test",
        "label_mode":   "int",
        "invert":       True,
        "augment_rotation": 12,       # RTL scripts tolerate more rotation
        "description":  "HODA Persian handwritten digits — 10 classes",
    },

    "arabic": {
        # ACHD (Arabic Characters Handwritten Dataset)  →  ~28–53 classes
        # Download: https://www.kaggle.com/datasets/mloey1/ahcd1
        # Comes as CSVs or images.  Adjust data_dir / label_mode accordingly.
        "num_classes":  28,           # 28 Arabic letters (base forms)
        "image_size":   32,
        "color_mode":   "grayscale",
        "stem_type":    "rtl",
        "data_dir":     "./data/arabic",
        "train_subdir": "Train",
        "test_subdir":  "Test",
        "label_mode":   "int",
        "invert":       True,
        "augment_rotation": 12,
        "description":  "ACHD Arabic characters — 28 classes",
    },

    # ── Logographic ───────────────────────────────────────────────────────────
    "kuzushiji49": {
        # Kuzushiji-49  →  49 hiragana classes (cursive historical Japanese)
        # Download: https://github.com/rois-codh/kmnist
        # Comes as .npz.
        "num_classes":  49,
        "image_size":   28,
        "color_mode":   "grayscale",
        "stem_type":    "symmetric",   # kanji/kana have balanced H+V strokes
        "data_dir":     "./data/kuzushiji49",
        "label_mode":   "npz",
        "npz_train":    "k49-train-imgs.npz",
        "npz_test":     "k49-test-imgs.npz",
        "npz_train_lbl":"k49-train-labels.npz",
        "npz_test_lbl": "k49-test-labels.npz",
        "npz_key":      "arr_0",       # key inside the .npz archive
        "invert":       True,
        "augment_rotation": 8,
        "description":  "Kuzushiji-49 — 49 hiragana classes",
    },

    # ── Syllabic ──────────────────────────────────────────────────────────────
    "thai": {
        # Burapha-TH Thai Handwritten Character Dataset
        # Download: contact Burapha University / IEEE DataPort
        # Expected: data/thai/Train/<class>/ data/thai/Test/<class>/
        "num_classes":  72,           # 44 consonants + 18 vowels + 10 digits
        "image_size":   32,
        "color_mode":   "grayscale",
        "stem_type":    "symmetric",   # Thai has circular loops + vertical strokes
        "data_dir":     "./data/thai",
        "train_subdir": "Train",
        "test_subdir":  "Test",
        "label_mode":   "int",
        "invert":       True,
        "augment_rotation": 10,
        "description":  "Burapha-TH Thai handwritten characters — 72 classes",
    },
}

# ─────────────────────────────────────────────────────────────────────────────
#  2. GLOBAL TRAINING CONFIG  (override per-run with CLI flags)
# ─────────────────────────────────────────────────────────────────────────────
TRAIN_CFG = {
    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,
    "results_dir":      "./results",
    "patience":         15,         # EarlyStopping patience
}

AUTOTUNE = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  3. DATASET LOADERS
# ─────────────────────────────────────────────────────────────────────────────

def load_directory_dataset(cfg: dict):
    """
    Standard keras.utils.image_dataset_from_directory loader.
    Works for: Devanagari, Bengali, Persian, Arabic, Thai, PG-HWLD.
    """
    IMG = cfg["image_size"]
    train_full = keras.utils.image_dataset_from_directory(
        os.path.join(cfg["data_dir"], cfg["train_subdir"]),
        image_size=(IMG, IMG), batch_size=None,
        color_mode=cfg["color_mode"], label_mode="int", seed=SEED,
    )
    test_raw = keras.utils.image_dataset_from_directory(
        os.path.join(cfg["data_dir"], cfg["test_subdir"]),
        image_size=(IMG, IMG), batch_size=None,
        color_mode=cfg["color_mode"], label_mode="int", seed=SEED,
    )
    return train_full, test_raw


def load_npz_dataset(cfg: dict):
    """
    Loader for datasets distributed as .npz or separate .npy files.
    Works for: Kannada-MNIST, Kuzushiji-49.
    """
    IMG   = cfg["image_size"]
    d_dir = cfg["data_dir"]
    key   = cfg.get("npz_key", "arr_0")

    def _load(fname):
        path = os.path.join(d_dir, fname)
        if fname.endswith(".npz"):
            return np.load(path)[key]
        return np.load(path)

    X_train = _load(cfg["npz_train"])
    X_test  = _load(cfg["npz_test"])
    y_train = _load(cfg["npz_train_lbl"])
    y_test  = _load(cfg["npz_test_lbl"])

    # Flatten label arrays if needed
    y_train = y_train.flatten().astype(np.int32)
    y_test  = y_test.flatten().astype(np.int32)

    # Ensure shape (N, H, W, 1)
    if X_train.ndim == 3:
        X_train = X_train[..., np.newaxis]
        X_test  = X_test[..., np.newaxis]

    # Resize if needed
    if X_train.shape[1] != IMG:
        X_train = tf.image.resize(X_train, [IMG, IMG]).numpy()
        X_test  = tf.image.resize(X_test,  [IMG, IMG]).numpy()

    train_ds = tf.data.Dataset.from_tensor_slices(
        (X_train.astype(np.float32), y_train))
    test_ds  = tf.data.Dataset.from_tensor_slices(
        (X_test.astype(np.float32),  y_test))

    return train_ds, test_ds


def load_tfds_dataset(cfg: dict):
    """
    Loader for datasets available via tensorflow_datasets.
    Works for: all EMNIST splits.
    """
    try:
        import tensorflow_datasets as tfds
    except ImportError:
        raise ImportError(
            "tensorflow-datasets is required for EMNIST.  "
            "Install it: pip install tensorflow-datasets"
        )
    IMG = cfg["image_size"]

    (train_raw, test_raw), info = tfds.load(
        cfg["tfds_name"],
        split=["train", "test"],
        with_info=True,
        as_supervised=True,   # yields (image, label)
        shuffle_files=True,
    )

    # EMNIST images are (28,28,1) but stored transposed; fix orientation.
    if cfg.get("transpose", False):
        def fix_orientation(img, lbl):
            img = tf.image.rot90(img, k=3)
            img = tf.image.flip_left_right(img)
            return img, lbl
        train_raw = train_raw.map(fix_orientation, num_parallel_calls=AUTOTUNE)
        test_raw  = test_raw.map(fix_orientation,  num_parallel_calls=AUTOTUNE)

    # Resize if needed
    if IMG != 28:
        def resize(img, lbl):
            return tf.image.resize(img, [IMG, IMG]), lbl
        train_raw = train_raw.map(resize, num_parallel_calls=AUTOTUNE)
        test_raw  = test_raw.map(resize,  num_parallel_calls=AUTOTUNE)

    return train_raw, test_raw


def load_dataset(cfg: dict):
    """
    Dispatch to the correct loader based on label_mode.
    Returns: (train_full_unbatched, test_unbatched)
    """
    mode = cfg["label_mode"]
    if mode == "int":
        return load_directory_dataset(cfg)
    elif mode == "npz":
        return load_npz_dataset(cfg)
    elif mode == "tfds":
        return load_tfds_dataset(cfg)
    else:
        raise ValueError(f"Unknown label_mode: {mode}")

# ─────────────────────────────────────────────────────────────────────────────
#  4. PREPROCESSING PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def build_pipeline(train_full, test_raw, cfg: dict, train_cfg: dict):
    """
    Splits train→(train, val), applies normalise + augment, returns
    (train_ds, val_ds, test_ds_int, test_ds_onehot, n_train, n_val)
    """
    IMG         = cfg["image_size"]
    NUM_CLASSES = cfg["num_classes"]
    BS          = train_cfg["batch_size"]
    invert      = cfg.get("invert", False)
    rot_deg     = cfg.get("augment_rotation", 10)

    # ── Val split ──────────────────────────────────────────────────────────
    total   = tf.data.experimental.cardinality(train_full).numpy()
    n_val   = max(1, int(total * train_cfg["val_split"]))
    n_train = total - n_val
    train_raw = train_full.take(n_train)
    val_raw   = train_full.skip(n_train)

    # ── Preprocessing helpers ──────────────────────────────────────────────
    def normalise(img, lbl):
        img = tf.cast(img, tf.float32)
        if invert:
            img = 255.0 - img
        img = img / 127.5 - 1.0          # → [-1, 1]
        return img, lbl

    def augment(img, lbl):
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        # Random translation via pad+crop
        img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
        img = tf.image.random_crop(img, [IMG, IMG, 1 if cfg["color_mode"] == "grayscale" else 3])
        # Random rotation (use tf.keras.layers.RandomRotation in a map for simplicity)
        # Approximated with a small shear for speed; replace with tfa.image.rotate if available.
        return img, lbl

    def to_onehot(img, lbl):
        return img, tf.one_hot(lbl, NUM_CLASSES)

    # ── Build pipelines ────────────────────────────────────────────────────
    train_ds = (
        train_raw
        .map(normalise,  num_parallel_calls=AUTOTUNE)
        .map(augment,    num_parallel_calls=AUTOTUNE)
        .map(to_onehot,  num_parallel_calls=AUTOTUNE)
        .shuffle(8192, seed=SEED)
        .batch(BS)
        .prefetch(AUTOTUNE)
    )
    val_ds = (
        val_raw
        .map(normalise,  num_parallel_calls=AUTOTUNE)
        .map(to_onehot,  num_parallel_calls=AUTOTUNE)
        .batch(BS)
        .prefetch(AUTOTUNE)
    )
    # Integer-label test set for macro-F1 calculation
    test_ds_int = (
        test_raw
        .map(normalise, num_parallel_calls=AUTOTUNE)
        .batch(BS)
        .prefetch(AUTOTUNE)
    )
    # One-hot test set for model.evaluate()
    test_ds_oh = (
        test_raw
        .map(normalise,  num_parallel_calls=AUTOTUNE)
        .map(to_onehot,  num_parallel_calls=AUTOTUNE)
        .batch(BS)
        .prefetch(AUTOTUNE)
    )

    print(f"[INFO] Train: {n_train:,} | Val: {n_val:,}")
    return train_ds, val_ds, test_ds_int, test_ds_oh, n_train, n_val

# ─────────────────────────────────────────────────────────────────────────────
#  5. BUILDING BLOCKS  (unchanged from original WhatNet)
# ─────────────────────────────────────────────────────────────────────────────

def relu(x):
    return tf.nn.relu(x)


def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(relu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(relu)(x)
    return x


def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(relu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    cat = layers.Concatenate()([r1, r2])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(relu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(relu)(out)
    return out


def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])


def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    h = layers.Dense(256, activation=relu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps = layers.BatchNormalization()(caps)
    return caps

# ─────────────────────────────────────────────────────────────────────────────
#  6. SCRIPT-AWARE STEM
#
#     stem_type controls which asymmetric kernels are used to initialise the
#     scaffold path.  The texture path (3×3) is always present.
#
#     "shirorekha"  →  1×5  horizontal  (Devanagari / Bengali top-bar)
#     "horizontal"  →  1×5  horizontal  (Latin — dominated by crossbars)
#     "rtl"         →  1×5 + 5×1 both  (Arabic / Persian — bidirectional joins)
#     "symmetric"   →  5×5              (Kuzushiji / Thai — balanced strokes)
#     "standard"    →  3×3 only         (digits, Kannada — no dominant direction)
# ─────────────────────────────────────────────────────────────────────────────

def build_script_stem(inp, stem_type: str):
    """
    Returns (stem_tensor, scaffold_tensor), each 32-channel feature maps.
    scaffold is reinjected at each encoder stage for stroke continuity.
    """
    # ── Texture path: always a standard 3×3 conv ─────────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t)
    t = layers.Activation(relu)(t)

    # ── Scaffold path: script-specific kernel(s) ──────────────────────────
    if stem_type in ("shirorekha", "horizontal"):
        # Single horizontal asymmetric kernel — captures the Devanagari
        # shirorekha (top connecting bar) and Latin crossbars alike.
        s = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
        s = layers.BatchNormalization()(s)
        scaffold = layers.Activation(relu)(s)

    elif stem_type == "rtl":
        # RTL scripts (Arabic, Persian) join characters and have strong
        # horizontal AND some vertical strokes depending on the letter form.
        # We use both directions and merge.
        sh = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
        sh = layers.BatchNormalization()(sh)
        sh = layers.Activation(relu)(sh)

        sv = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
        sv = layers.BatchNormalization()(sv)
        sv = layers.Activation(relu)(sv)

        scaffold = layers.Concatenate()([sh, sv])   # → 32 channels
        scaffold = layers.Conv2D(32, 1, use_bias=False)(scaffold)
        scaffold = layers.BatchNormalization()(scaffold)
        scaffold = layers.Activation(relu)(scaffold)

    elif stem_type == "symmetric":
        # Logographic / syllabic scripts have balanced strokes in both axes.
        # Use a square-ish kernel to avoid privileging any direction.
        s = layers.Conv2D(32, (3, 3), padding="same", use_bias=False)(inp)
        s = layers.BatchNormalization()(s)
        # Second branch with dilated conv for semi-local stroke context
        sd = layers.Conv2D(32, 3, padding="same",
                           dilation_rate=2, use_bias=False)(inp)
        sd = layers.BatchNormalization()(sd)
        scaffold = layers.Add()([layers.Activation(relu)(s),
                                  layers.Activation(relu)(sd)])
        scaffold = layers.Conv2D(32, 1, use_bias=False)(scaffold)
        scaffold = layers.BatchNormalization()(scaffold)
        scaffold = layers.Activation(relu)(scaffold)

    else:  # "standard"
        # No privileged direction — fall back to a plain 3×3 scaffold.
        s = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
        s = layers.BatchNormalization()(s)
        scaffold = layers.Activation(relu)(s)

    # ── Merge texture + scaffold ──────────────────────────────────────────
    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(relu)(stem)

    return stem, scaffold

# ─────────────────────────────────────────────────────────────────────────────
#  7. FULL MODEL  (our_model-Net generalised)
# ─────────────────────────────────────────────────────────────────────────────

def build_whatnet(
    num_classes: int,
    image_size:  int,
    stem_type:   str  = "shirorekha",
    color_mode:  str  = "grayscale",
    dropout_rate: float = 0.3,
    head_units:  int  = 256,
) -> Model:
    """
    WhatNet: script-agnostic handwriting recognition network.

    Architecture
    ────────────
    Stem (script-aware dual-path):
      • Texture path:  3×3 conv
      • Scaffold path: stem_type-specific asymmetric conv(s)
      → Merged with SE channel attention

    Encoder (3 stages, each halves spatial dims):
      enc1: 64→64   enc2: 64→128   enc3: 128→256
      Each stage injects a pooled scaffold residual (weight 0.1).

    Head:
      • Multi-scale GAP fusion  (enc1 + enc2 + enc3 → 64+128+256 = 448-d)
      • Adaptive Filter Capsule (AFC) → (B, K) class scores
      • Dense head              → (B, K) logits
      • Learnable gate          → blends AFC + dense logits
      → Final logits (B, K)

    Parameters change only: num_classes, image_size, stem_type.
    The encoder and head are fully identical across all scripts.
    """
    channels = 1 if color_mode == "grayscale" else 3
    inp = Input(shape=(image_size, image_size, channels), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    stem, scaffold = build_script_stem(inp, stem_type)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(
               layers.Conv2D(64, 1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(
               layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(
               layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Multi-scale GAP fusion ─────────────────────────────────────────────
    gap1 = layers.GlobalAveragePooling2D(name="gap1")(enc1)   # 64-d
    gap2 = layers.GlobalAveragePooling2D(name="gap2")(enc2)   # 128-d
    gap3 = layers.GlobalAveragePooling2D(name="gap3")(enc3)   # 256-d
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # 448-d

    # ── Adaptive Filter Capsule ────────────────────────────────────────────
    afc_out = adaptive_filter_capsule(fused_gap, num_classes)   # (B, K)

    # ── Dense head ─────────────────────────────────────────────────────────
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("relu", name="head_act")(x)
    x = layers.Dense(num_classes, name="head_logits")(x)

    # ── Gated fusion ───────────────────────────────────────────────────────
    combined = layers.Concatenate(name="gate_input")([x, afc_out])
    gate     = layers.Dense(2, activation="softmax", name="gate")(combined)

    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])

    outputs = layers.Add(name="logits")([x_scaled, afc_scaled])

    model_name = f"WhatNet-{stem_type}"
    return Model(inputs=inp, outputs=outputs, name=model_name)

# ─────────────────────────────────────────────────────────────────────────────
#  8. LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────

class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  9. COMPILE + TRAIN
# ─────────────────────────────────────────────────────────────────────────────

def compile_model(model: Model, steps_total: int, train_cfg: dict) -> Model:
    lr_sch = CosineAnnealing(train_cfg["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=train_cfg["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=train_cfg["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model


def compute_macro_f1(model: Model, dataset, num_classes: int) -> float:
    tp = np.zeros(num_classes)
    fp = np.zeros(num_classes)
    fn = np.zeros(num_classes)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(num_classes):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)


def train_and_evaluate(script_key: str, train_cfg: dict = TRAIN_CFG):
    """
    Full pipeline for a single script:
      load → preprocess → build → compile → train → evaluate → save
    """
    cfg = SCRIPT_CONFIGS[script_key]
    print(f"\n{'='*70}")
    print(f"  Script : {script_key}")
    print(f"  Desc   : {cfg['description']}")
    print(f"  Stem   : {cfg['stem_type']}  |  Classes: {cfg['num_classes']}")
    print(f"{'='*70}\n")

    os.makedirs(train_cfg["results_dir"], exist_ok=True)

    # ── Load data ──────────────────────────────────────────────────────────
    train_full, test_raw = load_dataset(cfg)

    # ── Build pipelines ────────────────────────────────────────────────────
    train_ds, val_ds, test_ds_int, test_ds_oh, n_train, n_val = build_pipeline(
        train_full, test_raw, cfg, train_cfg
    )

    # ── Build model ────────────────────────────────────────────────────────
    model = build_whatnet(
        num_classes = cfg["num_classes"],
        image_size  = cfg["image_size"],
        stem_type   = cfg["stem_type"],
        color_mode  = cfg["color_mode"],
    )

    steps_per_epoch = sum(1 for _ in train_ds)
    total_steps     = steps_per_epoch * train_cfg["epochs"]
    model = compile_model(model, total_steps, train_cfg)

    model.summary(line_length=80)

    # ── Train ──────────────────────────────────────────────────────────────
    ckpt_path = os.path.join(
        train_cfg["results_dir"], f"{script_key}_best.keras")
    callbacks = [
        ModelCheckpoint(
            ckpt_path, monitor="val_accuracy",
            save_best_only=True, verbose=0,
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=train_cfg["patience"],
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    print(f"\n  Training {script_key} …")
    t0 = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=train_cfg["epochs"],
        callbacks=callbacks,
        verbose=1,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(f"\n  Done: best val acc = {best_val:.2f}%  |  time = {elapsed:.0f}s")

    # ── Evaluate ───────────────────────────────────────────────────────────
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc = test_acc_raw * 100.0
    macro_f1 = compute_macro_f1(model, test_ds_int, cfg["num_classes"])

    results = {
        "script":    script_key,
        "stem_type": cfg["stem_type"],
        "classes":   cfg["num_classes"],
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "test_loss": round(float(test_loss), 4),
        "params":    model.count_params(),
        "train_sec": round(elapsed, 1),
        "best_val":  round(best_val, 2),
    }

    print(f"\n  Test Acc : {test_acc:.2f}%  |  Macro-F1 : {macro_f1:.2f}%")

    # ── Persist ────────────────────────────────────────────────────────────
    res_path = os.path.join(
        train_cfg["results_dir"], f"{script_key}_results.json")
    with open(res_path, "w") as f:
        json.dump(results, f, indent=2)

    hist_path = os.path.join(
        train_cfg["results_dir"], f"{script_key}_history.json")
    with open(hist_path, "w") as f:
        json.dump(
            {k: [float(v) for v in vals]
             for k, vals in history.history.items()},
            f, indent=2,
        )

    print(f"  Results  → {res_path}")
    print(f"  History  → {hist_path}")
    return model, results, history

# ─────────────────────────────────────────────────────────────────────────────
#  10. TRANSFER LEARNING HELPER
#
#  If you've already trained on one script and want to fine-tune on another
#  (e.g. Devanagari → Bengali, since both use a shirorekha stem), load the
#  source weights and freeze the encoder — only the head gets retrained.
# ─────────────────────────────────────────────────────────────────────────────

def finetune_from(
    source_script:  str,
    target_script:  str,
    source_weights: str,          # path to saved .keras model
    freeze_encoder: bool = True,
    train_cfg: dict = TRAIN_CFG,
):
    """
    Transfer encoder weights from source_script to target_script.

    freeze_encoder=True : only the classification head is trained (fast).
    freeze_encoder=False: full fine-tuning with a lower LR (recommended
                          when scripts differ significantly).

    Best pairings (shared stem type):
      devanagari  → bengali          (both "shirorekha")
      emnist_*    → pghwld           (both "horizontal")
      arabic      → persian          (both "rtl")
      kuzushiji49 → thai             (both "symmetric")
    """
    src_cfg = SCRIPT_CONFIGS[source_script]
    tgt_cfg = SCRIPT_CONFIGS[target_script]

    print(f"\n  Transfer: {source_script} → {target_script}")
    if src_cfg["stem_type"] != tgt_cfg["stem_type"]:
        print(f"  [WARN] stem types differ "
              f"({src_cfg['stem_type']} vs {tgt_cfg['stem_type']}). "
              f"Only encoder weights will transfer; stem is reinitialized.")

    # Build target model
    target_model = build_whatnet(
        num_classes = tgt_cfg["num_classes"],
        image_size  = tgt_cfg["image_size"],
        stem_type   = tgt_cfg["stem_type"],
        color_mode  = tgt_cfg["color_mode"],
    )

    # Load source model and copy matching layer weights by name
    source_model = keras.models.load_model(source_weights, compile=False)

    src_weights  = {w.name: w for w in source_model.weights}
    transferred  = 0
    for w in target_model.weights:
        if w.name in src_weights:
            src_w = src_weights[w.name]
            if w.shape == src_w.shape:
                w.assign(src_w)
                transferred += 1

    print(f"  Transferred {transferred} weight tensors.")

    # Optionally freeze the encoder layers
    if freeze_encoder:
        ENCODER_PREFIXES = ("dense_res_block", "residual_block",
                            "batch_normalization", "conv2d",
                            "depthwise_conv2d", "channel_attention")
        for layer in target_model.layers:
            if any(layer.name.startswith(p) for p in ENCODER_PREFIXES):
                layer.trainable = False
        frozen = sum(1 for l in target_model.layers if not l.trainable)
        print(f"  Frozen {frozen} layers. Fine-tuning head only.")

    return target_model

# ─────────────────────────────────────────────────────────────────────────────
#  11. CLI ENTRYPOINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="WhatNet — multi-script handwriting recognition")
    parser.add_argument(
        "--script", required=True,
        choices=list(SCRIPT_CONFIGS.keys()),
        help="Which script/dataset to train on",
    )
    parser.add_argument(
        "--epochs", type=int, default=TRAIN_CFG["epochs"],
        help="Max training epochs (default: 100)",
    )
    parser.add_argument(
        "--batch_size", type=int, default=TRAIN_CFG["batch_size"],
        help="Batch size (default: 64)",
    )
    parser.add_argument(
        "--lr", type=float, default=TRAIN_CFG["lr"],
        help="Peak learning rate (default: 5e-4)",
    )
    parser.add_argument(
        "--results_dir", type=str, default=TRAIN_CFG["results_dir"],
        help="Directory for checkpoints + JSON results",
    )
    # Transfer learning flags
    parser.add_argument(
        "--finetune_from", type=str, default=None,
        help="Source script key to transfer weights from",
    )
    parser.add_argument(
        "--source_weights", type=str, default=None,
        help="Path to source .keras model (required if --finetune_from set)",
    )
    parser.add_argument(
        "--freeze_encoder", action="store_true",
        help="Freeze encoder when fine-tuning (head-only training)",
    )

    args = parser.parse_args()

    cfg = dict(TRAIN_CFG)
    cfg["epochs"]      = args.epochs
    cfg["batch_size"]  = args.batch_size
    cfg["lr"]          = args.lr
    cfg["results_dir"] = args.results_dir

    if args.finetune_from:
        if not args.source_weights:
            parser.error("--source_weights is required when using --finetune_from")
        pretrained = finetune_from(
            source_script  = args.finetune_from,
            target_script  = args.script,
            source_weights = args.source_weights,
            freeze_encoder = args.freeze_encoder,
            train_cfg      = cfg,
        )
        # Load data for target script and continue training
        tgt_cfg = SCRIPT_CONFIGS[args.script]
        train_full, test_raw = load_dataset(tgt_cfg)
        train_ds, val_ds, test_ds_int, test_ds_oh, _, _ = build_pipeline(
            train_full, test_raw, tgt_cfg, cfg)
        steps = sum(1 for _ in train_ds) * cfg["epochs"]
        pretrained = compile_model(pretrained, steps, cfg)
        pretrained.fit(train_ds, validation_data=val_ds,
                       epochs=cfg["epochs"], verbose=1)
    else:
        train_and_evaluate(args.script, cfg)

I'll help you adapt the architecture for multiple scripts/datasets. Let me create a comprehensive, unified training framework.Here's a comprehensive unified framework that adapts the `our_model-Net` architecture across all your target scripts and datasets.The key insight is that only two things change between scripts: the **stem** (which kernel shapes capture that script's distinctive strokes) and **num_classes**. The encoder, AFC capsule, and gated fusion are fully reusable. Here's the complete unified codebase:Here's what changed and why, script by script:

**The core idea:** Only two things vary per script — `stem_type` and `num_classes`. The entire encoder (dense residual blocks, scaffold injection, multi-scale GAP, AFC capsule, gated fusion) is architecturally identical across all scripts, making weights transferable between related families.

**Stem types explained:**

- `shirorekha` (1×5 horizontal) — Devanagari and Bengali both have the distinctive top connecting bar (शिरोरेखा / মাত্রা). The original kernel was perfect here and stays unchanged.
- `horizontal` (same 1×5) — Latin scripts are dominated by crossbars and serifs, so the same kernel happens to work well, just for different reasons.
- `rtl` (1×5 + 5×1 merged) — Arabic and Persian join characters in both directions depending on letter position (initial/medial/final forms), so both H and V asymmetric kernels are needed.
- `symmetric` (3×3 + dilated 3×3 summed) — Kuzushiji-49 and Thai have balanced stroke complexity in all directions with no single dominant axis.
- `standard` (plain 3×3) — Kannada digits and similar compact datasets have no strong directional bias.

**Dataset loader routing** (`label_mode` field) handles three formats automatically: directory-based image folders (Devanagari, Bengali, Persian, Arabic, Thai, PG-HWLD), `.npz`/`.npy` arrays (Kannada-MNIST, Kuzushiji-49), and tensorflow-datasets (all 6 EMNIST splits).

**EMNIST gotcha** — the images are stored transposed by convention in the raw dataset. The `transpose: True` flag in each EMNIST config applies a rot90+flip to correct orientation before training.

**Transfer learning** — the `finetune_from()` helper copies weights by tensor name, so Devanagari → Bengali, EMNIST → PG-HWLD, Arabic → Persian, and Kuzushiji → Thai all work naturally since they share the same stem type. Use `--freeze_encoder` for fast head-only fine-tuning when scripts are closely related.

Run any script with:
```bash
python whatnet_multiscript.py --script bengali
python whatnet_multiscript.py --script emnist_balanced
python whatnet_multiscript.py --script arabic --finetune_from persian \
    --source_weights results/persian_best.keras --freeze_encoder
```